# Model Application on Uropod Dataset

This dataset is, from the Pixelgen paper, consists of T cells stimulated to induce uropod information.

This is the first dataset in which we show the use of colocalization, and in fact we use the Hotspot polarization & colocalization methods instead of the precomputed methods. Notice that feature selection of colocalization is paramount here because of the large number of features compared to samples. We select features by autocorrelations of colocalization pairs in (colocalization) PCA space.

The quality of the data is probably not very good. This is the first version of MPX (unlike the PBMCs and carT datasets which are version 2.0), which we were told is not great. Also the QC metrics (e.g. the tau metric) show significant problems and the number of samples is small. 

The model succeeds in modelling abundance and colocalization separately, and also in jointly modelling them (again - mostly the shared encoder models), but there is some price in the modelling of the abundance features (as seen by the higher errors as well as the slightly incompatibilities in the feature histograms). When training only abundance-colocalization (i.e. removing polarization) metrics are a bit better.

In [ ]:
import anndata
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune

from pathlib import Path


from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
from pixelator.statistics import clr_transformation
from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 


import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl



# from cytovi import CytoVI

print(torch.cuda.is_available())


scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
CMAP = 'RdBu_r'
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
%load_ext autoreload
%autoreload 2

In [ ]:
DATA_DIR = Path('./PixelGen/datasets/uropod')


FILENAMES = [
    "Uropod_CD54_fixed.layout.dataset.pxl",
    "Uropod_CD54_fixed_RANTES_stimulated.layout.dataset.pxl",
    "Uropod_CD54_fixed_MCP1_stimulated.layout.dataset.pxl",
    "Uropod_control.layout.dataset.pxl",
    "Uropod_control_RANTES_stimulated.layout.dataset.pxl",
    "Uropod_control_MCP1_stimulated.layout.dataset.pxl",
]

SAMPLE_NAMES = [
    "imm_no_cytokine", 
    "imm_rantes",
    "imm_mcp1", 
    "sol_no_cytokine",
    "sol_rantes",
    "sol_mcp1"
]

COMBINED_FILENAME = "uropod_combined.pxl"
COMBINED_PATH = DATA_DIR / COMBINED_FILENAME

pg_data = pixelator.read(COMBINED_PATH)

# Uncomment to download for first time

# BASEURL = "https://pixelgen-technologies-datasets.s3.eu-north-1.amazonaws.com/mpx-datasets/pixelator/0.18.x/uropod-t-cells-v1.0-immunology-I"
# pg_data = download_pxl(
#     baseurl=BASEURL,
#     filenames=FILENAMES,
#     sample_names=SAMPLE_NAMES,
#     dataset_dir=DATA_DIR,
#     dataset_full_path=COMBINED_PATH,
# )

## Annotation and PP
Scroll down to start from the preprocessed anndata

In [ ]:
adata = pg_data.adata.copy()
adata.raw = adata.copy()
sc.pp.calculate_qc_metrics(adata, percent_top=None, inplace=True, var_type='proteins')
adata.layers['counts'] = adata.X.copy()

In [ ]:
sc.pl.violin(
    adata,
    ["n_proteins_by_counts", "total_counts",],
    groupby='sample',
    jitter=0.3,
    multi_panel=True,
)
fig, ax = cell_count_plot(adata.obs, color_by="sample")
ax.tick_params(axis='x', labelrotation=45)

In [ ]:
molecule_rank_df = adata.obs[["sample", "molecules"]].copy()
molecule_rank_df["rank"] = molecule_rank_df.groupby(["sample"])["molecules"].rank(
    ascending=False, method="first"
)
fig_intersection, ax = molecule_rank_plot(molecule_rank_df, group_by="sample")
low = 10000
ax.axhline(y=low)

In [ ]:
tau_metrics_df = adata.obs[["sample", "tau", "mean_molecules_per_a_pixel", "tau_type"]]
tau_metrics_df = tau_metrics_df.rename(columns={"mean_molecules_per_a_pixel": "umi_per_upia"})


fig, ax = scatter_umi_per_upia_vs_tau(tau_metrics_df, group_by="sample")

In [ ]:
components = adata.obs[
    (adata.obs['tau_type'] == 'normal') & 
    (adata.obs['molecules'] > low)
].index
orig_adata = adata.copy()
adata = orig_adata[components, :]
cells_per_sample_df = (
    adata.obs.groupby("sample").size().to_frame(name="size").reset_index()
)

fig, ax = cell_count_plot(adata.obs, color_by="sample")

In [ ]:
ax = sc.pl.highest_expr_genes(adata, n_top=20, show=False)
ax.set_title('Highly Abundant Antibodies')
stats = adata.to_df().agg(['mean', 'var',], axis=0).T
fig, ax = plt.subplots(1)
sns.scatterplot(x=stats['mean'], y=stats['var'], ax=ax)
ax.loglog()
# var_genes = sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=20, batch_key='sample', layer='counts', inplace=False)
# sc.pl.highly_variable_genes(var_genes, show=True, log=True)

In [ ]:
adata.obs['sample'].unique()

In [ ]:
adata.obs['condition'] = adata.obs['sample'].map(
    {
        'imm_no_cytokine': 'control',
        'imm_rantes': 'imm_rantes',
        'imm_mcp1': 'imm_mcp1',
        'sol_no_cytokine': 'control',
        'sol_rantes': 'control',
        'sol_mcp1': 'control',
    }
)
imm_rantes_control_obs = adata.obs[adata.obs['sample'].isin(('imm_rantes', 'sol_no_cytokine'))].index

In [ ]:
isotype_controls=['mIgG1', 'mIgG2a', 'mIgG2b']
non_isotype_vars = [var for var in adata.var_names if var not in isotype_controls]

In [ ]:
adata.layers['dsb'] = dsb_normalize(adata.to_df('counts'), isotype_controls=isotype_controls)
adata.layers['clr'] = clr_transformation(adata.to_df('counts'), axis=1)
adata.layers['clr_by_ab'] = clr_transformation(adata.to_df('counts'), axis=0)
adata.layers['log1p'] = np.log1p(adata.to_df('counts'))

In [ ]:
sc.tl.pca(adata, layer='clr')
plot_cumulative_variance(adata)

In [ ]:
sc.pp.neighbors(adata,)
sc.tl.umap(adata, )
# Do not rerun or annotations will change
# sc.tl.leiden(adata, random_state=0)
# sc.tl.leiden(adata, restrict_to=('leiden', ['3']), resolution=0.5, key_added='leiden_R', random_state=0)
# sc.tl.leiden(adata, restrict_to=('leiden_R', ['9']), resolution=0.5, key_added='leiden_R', random_state=0)
leiden_key = 'leiden_R'
sc.pl.umap(adata, layer='clr', color=['sample', 'log1p_total_counts', leiden_key, 'CD3E', 'CD4', 'CD8', 'CD161', 'CD41'])

In [ ]:
cell_type_dict = {
    '0': 'Cytotoxic_T', 
    '1': 'Cytotoxic_T', 
    '4': 'Cytotoxic_T', 
    '6': 'Cytotoxic_T', 
    '10': 'Cytotoxic_T', 
    '11': 'Cytotoxic_T', 

    '2': 'Helper_T',
    '7': 'Helper_T',

    '5': 'Cytotoxic_T (CD8-CD161+)',
    '8': 'Cytotoxic_T (CD8-CD161+)',
    
    '9,1': 'Suspected_doublets',    # High counts, CD8+CD4+

    '3,1': 'Non_T', # CD41 
    '3,0': 'Non_T', # CD41 
    '3,2': 'Non_T', # CD41
    '3,3': 'Non_T', # CD41
    '9,0': 'Non_T',
    '9,2': 'Non_T',
    '9,3': 'Non_T'
}
adata.obs['cell_type'] = adata.obs[leiden_key].map(cell_type_dict)
sc.pl.umap(adata,  color=['condition', 'cell_type',])

In [ ]:
adata = adata[adata.obs[adata.obs['cell_type'] != 'Non_T'].index,:].copy()
print(adata.obs['cell_type'].unique())

In [ ]:
vars = [v for v in adata.var_names if v not in ['mIgG1', 'mIgG2a', 'mIgG2b']]

# Pixelgen pol & coloc

polarization_i = convert_polarization_to_feature_matrix(pg_data.polarization, components=adata.obs.index, key='morans_i', vars=vars)
polarization_z = convert_polarization_to_feature_matrix(pg_data.polarization, components=adata.obs.index, key='morans_z', vars=vars)

coloc_p = convert_colocalization_to_feature_matrix(pg_data.colocalization, components=adata.obs.index, key='pearson', vars=vars)
coloc_z = convert_colocalization_to_feature_matrix(pg_data.colocalization, components=adata.obs.index, key='pearson_z', vars=vars)
adata.obsm['pol_i'] = polarization_i
adata.obsm['pol_z'] = polarization_z
adata.obsm['coloc_p'] = coloc_p
adata.obsm['coloc_z'] = coloc_z


## After PP

In [ ]:
# adata.write_h5ad(DATA_DIR / 'uropod_combined_annotated_with_pol.h5ad')
adata = anndata.read_h5ad(DATA_DIR / 'uropod_combined_annotated_with_pol.h5ad')

## Hotspot Pol & Coloc

In [ ]:
# Hotspot pol and coloc

unfiltered_with_pol = anndata.read_h5ad(DATA_DIR / 'uropod_combined_with_pol.h5ad')

for key in ('pol_hs_c', 'pol_hs_z', 'coloc_hs_c', 'coloc_hs_z'):
    adata.obsm[key] = unfiltered_with_pol.obsm[key].loc[adata.obs.index]
    if 'pol' in key:
        adata.obsm[key].columns = [f'{c}_pol' for c in adata.obsm[key].columns]

## Coloc Feature Selection
Coloc has protein^2 features - too many for our low-observation regime, and they have a lot of redundant information.
We select pairs which correlate highly in coloc PCA space, and for which neither of the markers is a highly abundant protein.

In [ ]:
n_coloc_features = 50
calc_PCA(adata, rep='coloc_hs_c', key_added='coloc_hs_c')
coloc_autocorr = distr_autocorrelation_in_latent(adata, latent_keys=['coloc_hs_c_pca'], names=['coloc_hs_c_pca'],
                                                 rep_key='coloc_hs_c'
                                                 )
blacklist = ['HLA-ABC', 'B2M']

coloc_autocorr = split_pair_column(coloc_autocorr.rename_axis('pair').reset_index(), c='pair')
coloc_autocorr = filter_df_by_two_columns(coloc_autocorr, c1='marker_1', c2='marker_2', blacklist=blacklist)

coloc_autocorr.sort_values(by='morans', ascending=False, inplace=True)

ax = rank_plot(coloc_autocorr['morans'], s=5)
ax.axvline(x=n_coloc_features)
coloc_hvg_vars = coloc_autocorr['pair'][:n_coloc_features]

print(list(coloc_hvg_vars))

In [ ]:
n_pol_features = 20

calc_PCA(adata, rep='pol_hs_c', key_added='pol_hs_c')
pol_autocorr = distr_autocorrelation_in_latent(adata, latent_keys=['pol_hs_c_pca'], names=['pol_hs_c_pca'],
                                                 rep_key='pol_hs_c'
                                                 )
pol_hvg_vars = pol_autocorr.sort_values(by='morans', ascending=False).index[:n_pol_features]
print(list(pol_hvg_vars))

In [ ]:
# Instead of standardizing each feature separately, it worked better to choose a uniform scaling factor, based on 
# knowledge of the data, to bring the features  into unit values (when standardizing separately, 
# there is no difference in magnitude between less meaningful and more meaningful features)

adata.obsm['pol_hvg'] = adata.obsm['pol_hs_c'][pol_hvg_vars]
adata.obsm['coloc_hvg'] = adata.obsm['coloc_hs_c'][coloc_hvg_vars]

# adata.obsm['pol_hvg_std'] = std_clip(standardize(adata.obsm['pol_hvg']), c=3)
# adata.obsm['coloc_hvg_std'] = std_clip(standardize(adata.obsm['coloc_hvg']), c=3)

# adata.obsm['pol_std'] = std_clip(standardize(adata.obsm['pol_hs_c']), c=3)
# adata.obsm['coloc_std'] = std_clip(standardize(adata.obsm['coloc_hs_c']), c=3)

scale_factor = 10

adata.obsm['pol_hvg_std'] = scale_factor*adata.obsm['pol_hvg']
adata.obsm['coloc_hvg_std'] = scale_factor*adata.obsm['coloc_hvg']

adata.obsm['pol_std'] = scale_factor*adata.obsm['pol_hs_c']
adata.obsm['coloc_std'] = scale_factor*adata.obsm['coloc_hs_c']


print(f'Highly Variable Pol: {adata.obsm["pol_hvg"].shape[1]}/{adata.obsm["pol_hs_c"].shape[1]}')
print(f'Highly Variable Coloc: {adata.obsm["coloc_hvg"].shape[1]}/{adata.obsm["coloc_hs_c"].shape[1]}')

In [ ]:
ab_layer = 'clr'
pol_key = 'pol_hvg_std'
coloc_key = 'coloc_hvg_std'

In [ ]:
adata.obsm['ab_pol_coloc'] = pd.concat(
    (adata.to_df(ab_layer), adata.obsm[pol_key], adata.obsm[coloc_key]),
    axis=1,
)
calc_PCA(adata, rep='ab_pol_coloc', key_added='ab_pol_coloc')
calc_PCA(adata, rep=ab_layer, key_added=ab_layer)
calc_PCA(adata, rep=coloc_key, key_added=coloc_key)

## PCA

Even abundance-only pretty much higlights the difference in conditions, however when introducing pol and coloc this takes on the most importance (but the difference between T Helpers and Cytotoxic cells is preserved).

In [ ]:
adata.obs['(CD162,CD50)'] = adata.obsm[coloc_key]['(CD162,CD50)']
_ = pca_neighbors_umap(adata, ab_layer, umap_pl_kwargs=dict(layer=ab_layer, color=['condition', 'cell_type', '(CD162,CD50)'], vcenter=0, cmap=CMAP), umap_title='Abundance Only')
_ = pca_neighbors_umap(adata, 'ab_pol_coloc', umap_pl_kwargs=dict(layer=ab_layer, color=['condition', 'cell_type','(CD162,CD50)',], vcenter=0, cmap=CMAP), umap_title='Abundance + Pol + Coloc')
_ = pca_neighbors_umap(adata, coloc_key, umap_pl_kwargs=dict(layer=ab_layer, color=['condition', 'cell_type','(CD162,CD50)',], vcenter=0, cmap=CMAP), umap_title='Coloc Only')

## Abundance Model

In [ ]:
model_cls = MultiModalSCVI
setup_kwargs = dict(layer=ab_layer, batch_key=None)
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=3e-4, optimizer='Adam', n_epochs_kl_warmup=400))
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=1, dropout_rate=0.1, distrs=[D.Normal,],)
modalities_latent_names=[(ab_layer, 'ab_model')]
abundance_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)
get_model_latents(adata, abundance_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type', '(CD162,CD50)',], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## Polarization-only

In [ ]:
obs = adata.obs.copy()
try:
    obs.drop(columns='CD50_pol', inplace=True)
except:
    pass
pol_hvg_adata = anndata.AnnData(
    X=adata.obsm['pol_hvg_std'],
    obs=obs,
    layers={
        'pol_hvg': adata.obsm['pol_hvg'],
        'pol_hvg_std': adata.obsm['pol_hvg_std'],
    }
)

In [ ]:
model_cls = MultiModalSCVI

setup_kwargs = dict(layer=pol_key, extra_modality_keys=[], n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=10, n_hidden=128, n_layers=1, dropout_rate=0.1, 
                        distrs=[D.Normal], 
                        joint_kl=False, 
                        unimodal_kl=True,
                        external_kl_weight=1, 
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400))

latent_name = 'pol_model'
modalities_latent_names=[(pol_key, latent_name)]
pol_model = train_model(pol_hvg_adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs)
get_model_latents(pol_hvg_adata, pol_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(pol_hvg_adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type', 'CD50_pol', 'CD162_pol'], vcenter=0, cmap='RdBu_r', layer=pol_key)).suptitle(title)

## Colocalization-only

In [ ]:
obs = adata.obs.copy()
try:
    obs.drop(columns='(CD162,CD50)', inplace=True)
    obs.drop(columns='(CD137,HLA-ABC)', inplace=True)
except:
    pass

coloc_adata = anndata.AnnData(
    X=adata.obsm['coloc_std'],
    obs=obs,
    layers={
        'coloc_std': adata.obsm['coloc_std'],
        'coloc_c': adata.obsm['coloc_hs_c'],
    }
)

coloc_hvg_adata = anndata.AnnData(
    X=adata.obsm['coloc_hvg_std'],
    obs=obs,
    layers={
        'coloc_hvg': adata.obsm['coloc_hvg'],
        'coloc_hvg_std': adata.obsm['coloc_hvg_std'],
    }
)

In [ ]:
model_cls = MultiModalSCVI

cur_adata = coloc_hvg_adata
layer = 'coloc_hvg_std'

setup_kwargs = dict(layer=layer, extra_modality_keys=[], n_modalities=1, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal], 
                        joint_kl=False, unimodal_kl=True,
                        external_kl_weight=1, 
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_monitor='elbo_validation',
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                    )

latent_name = 'coloc_model'
modalities_latent_names=[(layer, latent_name)]
coloc_model = train_model(cur_adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs)
get_model_latents(cur_adata, coloc_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(cur_adata, name, 
        umap_pl_kwargs=dict(color=['condition', 'cell_type', '(CD162,CD50)'], vcenter=0, cmap='RdBu_r', layer=layer)).suptitle(title)

## Abundance + Pol + Coloc Model

### Separate Encoders, Learned Global Weights

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'learned_weights'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[pol_key, coloc_key], n_modalities=3, batch_key=None, )
model_kwargs = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal, D.Normal], 
                        agg_method=AggMethod.AOE_GLOBAL_WEIGHTS,
                        joint_kl=False, unimodal_kl=True,
                        loss_weights='auto',
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp'),
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
modalities_latent_names=[
    ('joint', latent_name), (ab_layer, f'{latent_name}_{ab_layer}'), 
    (pol_key, f'{latent_name}_{pol_key}'), (coloc_key, f'{latent_name}_{coloc_key}')
]
learned_weights_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)
weights = learned_weights_model.get_weights()
get_model_latents(adata, learned_weights_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name if key != 'joint' else f'{name}, weights: {np.array2string(weights, precision=2)}'    
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type', '(CD162,CD50)',], vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## Shared Encoder

### Abundance-coloc only

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'ab_coloc_model'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[coloc_key], n_modalities=2, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal,], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        loss_weights='auto',
                        joint_kl=True, unimodal_kl=False,
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
ab_coloc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, ab_coloc_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type', '(CD162,CD50)',], 
                                                              vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

### All three

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'shared_enc_model'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[pol_key, coloc_key], n_modalities=3, batch_key=None, )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        loss_weights='auto',
                        joint_kl=True, unimodal_kl=False,
                        external_kl_weight=1,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type', '(CD162,CD50)',], 
                                                              vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

### Shared Encoder - Using batch key

In [ ]:
model_cls = MultiModalSCVI

latent_name = 'shared_enc_batch_key_model'

setup_kwargs = dict(layer=ab_layer, extra_modality_keys=[pol_key, coloc_key], n_modalities=3, batch_key='sample', )
model_kwargs = dict(n_latent=30, n_hidden=128, n_layers=2, dropout_rate=0.1, 
                        distrs=[D.Normal, D.Normal, D.Normal], 
                        agg_method=AggMethod.SHARED_ENCODER,
                        loss_weights='auto',
                        joint_kl=True, unimodal_kl=False,
                        decoder_kwargs=dict(decoder_param_eps=1e-2, decoder_activation='exp')
                    )
train_kwargs = dict(train_size=0.8, check_val_every_n_epoch=1, early_stopping=True, 
                    early_stopping_patience=200, batch_size=2000,
                    max_epochs=10000, enable_checkpointing=True, 
                    plan_kwargs=dict(lr=1e-4, optimizer='Adam', n_epochs_kl_warmup=400)
                )
shared_enc_batch_key_model = train_model(adata, model_cls=model_cls, setup_kwargs=setup_kwargs, model_kwargs=model_kwargs, train_kwargs=train_kwargs,)

modalities_latent_names=[('joint', latent_name)]
get_model_latents(adata, shared_enc_batch_key_model, modalities_latent_names=modalities_latent_names)
for key, name in modalities_latent_names:
    title = name
    fig = pca_neighbors_umap(adata, name, umap_pl_kwargs=dict(color=['condition', 'cell_type', '(CD162,CD50)',], 
                                                              vcenter=0, cmap='RdBu_r', layer=ab_layer)).suptitle(title)

## Metrics

In [ ]:
add_one_hot_encoding_obsm(adata, obs_column='cell_type')

In [ ]:
metrics = MultiModalVIMetrics(
    adata,
    models = {
        'abundance_only': abundance_model,
        'global_weights': learned_weights_model,
        'shared_enc': shared_enc_model,
        'shared_enc_ab_coloc': ab_coloc_model,
        'shared_enc_w_batch': shared_enc_batch_key_model,
        'pol_model': pol_model,
        'coloc_model': coloc_model
    },
    pca_key='ab_pol_coloc_pca',
    additional_autocorr_keys=['cell_type'],
)
metrics.run()

In [ ]:
_ = metrics.mean_modality_errors_barplot()
_ = metrics.mean_modality_errors_barplot(reconstruction_mean=True)

In [ ]:
_ = metrics.mean_autocorr_barplot()
_ = metrics.autocorr_barplot(autocorr_key=pol_key, auto_filter_features=10)
_ = metrics.autocorr_barplot(autocorr_key=coloc_key, features=coloc_hvg_vars[:10])
_ = metrics.autocorr_barplot(autocorr_key='cell_type', auto_filter_features=10, figsize=(10, 8))

In [ ]:
_ = metrics.feature_histplot(modality=coloc_key, features=['(CD162,CD50)', '(CD37,CD50)'], hue='condition')
_ = metrics.feature_histplot(modality=ab_layer, features=['CD4', 'CD8'], hue='condition')

In [ ]:
_ = metrics.top_autocorr_features_barplot(key='coloc_hvg_std', model_names=['coloc_model', 'shared_enc_ab_coloc', 'ab_pol_coloc_pca'], top=10)